# ⚖️ Fine-Tuning Llama 3.1 8B on Indian Tax Law
### QLoRA + Unsloth | Google Colab T4 (Free)

**Pipeline overview:**
1. Install Unsloth + dependencies
2. Load Llama 3.1 8B in 4-bit (fits on 16GB T4)
3. Apply LoRA adapters
4. Build Indian Tax Law dataset (500 Q&A pairs)
5. Fine-tune with SFTTrainer
6. Evaluate base vs fine-tuned
7. Export to GGUF & push to HuggingFace Hub

**Estimated total time:** ~2 hours (45 min compute, rest is dataset prep)

**Cost:** Free (Colab T4 + HuggingFace free tier)

> ⚠️ **Disclaimer:** This model is for research and educational purposes only. It is NOT a substitute for professional tax advice from a qualified CA/CPA. Always verify outputs against the official Income Tax Act and GST notifications.

---
## 📦 Step 1: Install Unsloth & Dependencies
*Runtime: ~5 minutes. Run this cell first, then restart the runtime when prompted.*

In [ ]:
# Install Unsloth (handles Llama, Mistral, Gemma, Phi)
# This installs: unsloth, transformers, trl, peft, bitsandbytes, accelerate
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps xformers trl peft accelerate bitsandbytes --quiet

# Verify GPU
!nvidia-smi
print("\n✅ Installation complete!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 130.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 133.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.3 MB/s e

In [ ]:
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
import json

# Confirm CUDA is available
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


---
## 🤖 Step 2: Load Llama 3.1 8B in 4-bit (QLoRA)

**What's happening here (the analogy):**
Imagine compressing a 4K video file to a smaller format for storage — you lose minimal quality
but save 75% of disk space. That's what 4-bit quantization does to the model weights:
16GB → ~4.5GB VRAM. Then we add small trainable "sticky notes" (LoRA adapters) on top,
without touching the compressed original.

In [ ]:
# ─── Model configuration ───────────────────────────────────────────
MAX_SEQ_LENGTH = 2048   # Max tokens per training example (covers most tax Q&A)
DTYPE = None            # Auto-detect: bfloat16 on A100/H100, float16 on T4
LOAD_IN_4BIT = True     # QLoRA: compresses weights from 16-bit → 4-bit

# Load Llama 3.1 8B Instruct (already quantized via bitsandbytes)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

print(f"✅ Model loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

==((====))==  Unsloth 2026.5.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit as a legacy tokenizer.


✅ Model loaded!
Parameters: 4.5B
VRAM used: 5.75 GB


### Apply LoRA Adapters

**LoRA analogy:** Instead of rewriting an entire textbook (retraining 8B parameters), we add sticky notes
at key chapters (attention layers). The sticky notes are tiny (rank=16 matrices), but they change how the
model interprets and responds. We only train these ~0.5% of total parameters.

In [ ]:
# ─── LoRA Configuration ────────────────────────────────────────────
# r=16: rank of adapter matrices (higher = more capacity but slower)
# lora_alpha=32: scaling factor (usually 2x rank)
# lora_dropout=0: no dropout for deterministic training on small datasets

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention layers
        "gate_proj", "up_proj", "down_proj",        # MLP layers
    ],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth's optimized checkpointing
    random_state = 42,
    use_rslora = False,    # Rank-stabilized LoRA — False for simplicity
    loftq_config = None,
)

# Show trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA adapters applied!")
print(f"Trainable parameters: {trainable:,} ({100 * trainable / total:.2f}% of total)")
print(f"Frozen parameters: {total - trainable:,}")

Unsloth 2026.5.8 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA adapters applied!
Trainable parameters: 41,943,040 (0.92% of total)
Frozen parameters: 4,540,600,320


---
## 📚 Step 3: Build the Indian Tax Law Dataset

**The Alpaca format** — each training example has 3 fields:
- `instruction`: The question/task
- `input`: Additional context (can be empty)
- `output`: The expert answer (include section citations, calculations, and compliance steps)

**Quality checklist for each example:**
- [ ] Cites the relevant section (e.g., Section 80C, Section 194J)
- [ ] Shows the calculation if numbers are involved
- [ ] Mentions the compliance step (filing, deadlines, forms)
- [ ] Includes thresholds and limits
- [ ] Written from a CA/tax practitioner perspective

In [ ]:
# ─── Seed Dataset: 60 High-Quality Indian Tax Law Q&A Pairs ─────────
# These cover the most commonly asked / most hallucinated topics.
# Expand to 500+ by adding more examples in the same format.

TAX_QA_DATA = [
    # ── SECTION 1: TDS ──────────────────────────────────────────────
    {
        "instruction": "What is the TDS rate on professional fees under Section 194J and when is it applicable?",
        "input": "A company pays ₹45,000 to a freelance chartered accountant for audit services.",
        "output": "Under Section 194J of the Income Tax Act, TDS is deducted at 10% on professional fees paid to a resident. The threshold limit is ₹30,000 per financial year per payee. Since the payment is ₹45,000 (exceeds ₹30,000 threshold), TDS applies. Calculation: TDS = 10% × ₹45,000 = ₹4,500. Net payment to CA = ₹40,500. The deductor must: (1) Deposit ₹4,500 with the government by the 7th of the following month, (2) File Form 26Q quarterly, and (3) Issue Form 16A to the CA by the due date. Note: Technical services attract 2% TDS under 194J(1)(ba) since Budget 2020, but professional services (CA, doctor, lawyer) remain at 10%."
    },
    {
        "instruction": "What is the TDS rate on contractor payments under Section 194C?",
        "input": "A company pays ₹1,20,000 to an individual contractor for interior design work over the year.",
        "output": "Under Section 194C, TDS on contractor payments is: 1% if the payee is an individual or HUF, 2% if the payee is any other person (company, firm, etc.). The threshold is ₹30,000 per single payment OR ₹1,00,000 aggregate per financial year. Since the aggregate payment is ₹1,20,000 (exceeds ₹1,00,000), TDS must be deducted. Calculation: TDS = 1% × ₹1,20,000 = ₹1,200 (since payee is an individual). TDS should be deducted at the time of credit or payment (whichever is earlier). Deposit by 7th of next month. No TDS is required if the contractor provides a declaration under Form 15G (for non-taxable income) or if PAN is furnished and payment is below threshold."
    },
    {
        "instruction": "What is the TDS rate for interest on fixed deposits, and what is the threshold?",
        "input": "A senior citizen has a fixed deposit in SBI earning ₹55,000 interest in FY 2024-25.",
        "output": "Under Section 194A, TDS on interest (other than securities) is deducted at 10%. Thresholds for FY 2024-25: ₹40,000 for non-senior citizens, ₹50,000 for senior citizens (aged 60+). Since this is a senior citizen with ₹55,000 interest (exceeds ₹50,000), TDS applies. TDS = 10% × ₹55,000 = ₹5,500. However, if the senior citizen's total income is below the basic exemption limit, they can submit Form 15H to the bank to avoid TDS deduction. Banks issue Form 16A showing TDS deducted. The senior citizen can claim credit for this TDS in their ITR and may get a refund if their tax liability is lower than TDS deducted."
    },
    {
        "instruction": "What is the TDS rate on rent under Section 194I?",
        "input": "A company pays monthly rent of ₹80,000 for its office premises to an individual landlord.",
        "output": "Under Section 194I, TDS on rent is: 10% for rent of land, building, or furniture/fittings, 2% for rent of plant, machinery, or equipment. The threshold is ₹2,40,000 per financial year (₹20,000 per month × 12). Since the annual rent is ₹9,60,000 (₹80,000 × 12, far exceeds ₹2,40,000), TDS must be deducted. Monthly TDS = 10% × ₹80,000 = ₹8,000. The tenant (deductor) must deposit TDS by 7th of the next month and file Form 26Q quarterly. The landlord receives Form 16A and can claim TDS credit in their ITR. Note: From Budget 2024, TDS under 194IB (individuals/HUF paying rent >₹50,000/month) is 2%."
    },
    {
        "instruction": "What happens if TDS is deducted but not deposited to the government on time?",
        "input": "",
        "output": "Failure to deposit TDS on time attracts both interest and penalties: Interest under Section 201(1A): 1.5% per month (or part thereof) from the date of deduction to the date of deposit. Penalty under Section 271C: Equal to the amount of TDS not deducted or deposited (discretionary, levied by Assessing Officer). Prosecution under Section 276B: Rigorous imprisonment of 3 months to 7 years + fine (in severe cases). Late filing of TDS returns (Form 24Q/26Q) under Section 234E: ₹200 per day of delay, capped at TDS amount. Best practice: Set standing instructions to auto-transfer TDS to the government's tax account (OLTAS) by the 7th of each month (except March — deadline is April 30th for March deductions)."
    },

    # ── SECTION 2: INCOME TAX DEDUCTIONS ────────────────────────────
    {
        "instruction": "What investments qualify for deduction under Section 80C and what is the maximum limit?",
        "input": "",
        "output": "Section 80C allows deduction up to ₹1,50,000 per financial year from gross total income. Qualifying investments and payments include: (1) Life Insurance Premium (for self, spouse, children), (2) EPF contributions (both employee's share), (3) PPF contributions, (4) NSC (National Savings Certificate), (5) ELSS (Equity Linked Savings Scheme) — 3-year lock-in, (6) Sukanya Samriddhi Account (for girl child), (7) Senior Citizens Savings Scheme (SCSS), (8) 5-year fixed deposit with scheduled banks, (9) Home loan principal repayment, (10) Children's tuition fees (up to 2 children). Important: Section 80C is only available under the Old Tax Regime. Under the New Tax Regime (default from FY 2023-24), Section 80C deduction is NOT available. The ₹1.5L limit is a combined cap for all investments under 80C, 80CCC (pension), and 80CCD(1)."
    },
    {
        "instruction": "Can I claim HRA exemption and home loan interest deduction simultaneously?",
        "input": "A salaried employee in Mumbai pays rent of ₹25,000/month and also has a home loan on a flat in Pune (occupied by parents).",
        "output": "Yes, both can be claimed simultaneously in this case. HRA Exemption (Section 10(13A)): The employee lives in a rented house in Mumbai → HRA exemption applies. Exemption is the MINIMUM of: (1) Actual HRA received, (2) Rent paid minus 10% of basic salary, (3) 50% of basic salary (for metro cities like Mumbai). Home Loan Interest (Section 24(b)): The Pune flat is occupied by parents, so it qualifies as a self-occupied or let-out property. If treated as self-occupied: Deduction up to ₹2,00,000 per year. If treated as let-out: Full interest deductible, but rental income must also be declared. This dual claim is valid when the employee lives in a different city from the owned property for genuine employment reasons. Supporting documentation required: Rent receipts, rental agreement, home loan statement, and Form 16 from employer showing HRA component."
    },
    {
        "instruction": "What is the deduction available under Section 80D for health insurance premium?",
        "input": "A 45-year-old taxpayer pays ₹22,000 premium for self/spouse/children and ₹32,000 for parents (both parents are 68 years old).",
        "output": "Under Section 80D, deductions are: For self, spouse, and dependent children (non-senior citizens): Up to ₹25,000. For parents who are senior citizens (60+): Up to ₹50,000. Calculation for this case: Self/spouse/children premium: ₹22,000 (eligible, within ₹25,000 limit). Parents premium: ₹32,000 (eligible, within ₹50,000 senior citizen limit). Total 80D deduction: ₹22,000 + ₹32,000 = ₹54,000. Additional benefit: ₹5,000 of preventive health check-up expenses is included within the overall limit (not in addition to it). Note: Section 80D deduction is available under both Old and New Tax Regime (it's one of the few deductions allowed under the new regime). Premium must be paid by cheque/digital mode (cash payment for amounts >₹2,000 is not allowed, except for preventive health check-ups)."
    },
    {
        "instruction": "What is the additional deduction available under Section 80CCD(1B) for NPS?",
        "input": "",
        "output": "Section 80CCD(1B) provides an ADDITIONAL deduction of up to ₹50,000 per year for contributions to the National Pension System (NPS) Tier-I account. This is over and above the ₹1,50,000 limit under Section 80C. How it works: Section 80CCD(1): NPS contribution up to 10% of salary is included within the ₹1,50,000 cap of 80C. Section 80CCD(1B): An additional ₹50,000 exclusively for NPS, beyond the 80C cap. Combined maximum: ₹2,00,000 (₹1,50,000 under 80C + ₹50,000 under 80CCD(1B)). This makes NPS the only instrument with its own exclusive ₹50,000 deduction bucket. Important: Like 80C, this deduction is only available under the Old Tax Regime. Tax on NPS withdrawal: 60% lump sum at retirement is tax-free; 40% must be used to buy annuity (annuity income is taxable)."
    },

    # ── SECTION 3: CAPITAL GAINS ─────────────────────────────────────
    {
        "instruction": "What is the tax on Long Term Capital Gains (LTCG) on equity shares sold on a stock exchange?",
        "input": "A taxpayer sold listed equity shares in August 2024 and made a profit of ₹1,80,000. The shares were purchased 2 years ago.",
        "output": "Under Section 112A (introduced in Budget 2018), LTCG on equity shares traded on a recognized stock exchange (with STT paid) is taxed at 10% WITHOUT indexation, but only on gains exceeding ₹1,00,000 in a financial year. Calculation: Total LTCG = ₹1,80,000. Exempt amount = ₹1,00,000 (first ₹1L is exempt). Taxable LTCG = ₹1,80,000 - ₹1,00,000 = ₹80,000. Tax = 10% × ₹80,000 = ₹8,000. Add: 4% Health & Education Cess = ₹320. Total tax = ₹8,320. Holding period for LTCG on equity: More than 12 months. STT (Securities Transaction Tax) must have been paid on both purchase and sale (exchange transactions). If sold before 12 months, STCG at 15% under Section 111A applies. Note: From Budget 2024, LTCG rate increased from 10% to 12.5% and ₹1L exemption removed for sales after July 23, 2024."
    },
    {
        "instruction": "What is the capital gains tax on sale of a residential property?",
        "input": "A property purchased in 2015 for ₹30 lakhs is sold in 2024 for ₹90 lakhs. The owner wants to know the tax implications.",
        "output": "Sale of residential property is Long Term Capital Gain (LTCG) if held for more than 24 months. Tax rate under Section 112: 20% WITH indexation benefit (OR 12.5% without indexation from Budget 2024 — taxpayer can choose lower). With indexation (using Cost Inflation Index): Purchase year 2015-16 CII = 254. Sale year 2024-25 CII = 363 (approximate). Indexed cost = ₹30L × (363/254) = ₹42.87L. LTCG = ₹90L - ₹42.87L = ₹47.13L. Tax at 20% = ₹9.43L. Without indexation (Budget 2024 option): LTCG = ₹90L - ₹30L = ₹60L. Tax at 12.5% = ₹7.5L. Choose the lower tax option. Exemptions available: Section 54: Reinvest proceeds in another residential property within 2 years → LTCG exempt up to purchase cost of new property. Section 54EC: Invest in NHAI/REC bonds within 6 months → exempt up to ₹50L. Surcharge + Cess applies additionally."
    },
    {
        "instruction": "What is STCG on mutual funds, and how is it taxed?",
        "input": "An investor redeems equity mutual fund units held for 8 months and earns a profit of ₹40,000.",
        "output": "Short Term Capital Gains (STCG) on equity-oriented mutual funds (where equity component is >65%) sold before 12 months is taxed at 15% under Section 111A. From Budget 2024, the rate increased to 20% for sales after July 23, 2024. Calculation (post July 23, 2024 sale): STCG = ₹40,000. Tax = 20% × ₹40,000 = ₹8,000. Health & Education Cess = 4% × ₹8,000 = ₹320. Total tax = ₹8,320. STT condition: The fund must have paid STT at redemption (all equity MFs listed on exchanges do). Debt mutual funds: No STCG/LTCG benefit — taxed as per slab rate regardless of holding period (since April 2023). ELSS funds: 3-year lock-in, so always LTCG (now at 12.5% above ₹1.25L from Budget 2024). Grandfathering: For equity MF units purchased before Jan 31, 2018, cost is the higher of actual cost or NAV on Jan 31, 2018."
    },

    # ── SECTION 4: GST ───────────────────────────────────────────────
    {
        "instruction": "What are the GST rates applicable in India and which category do essential items fall under?",
        "input": "",
        "output": "India's GST has a 5-slab structure: 0% (Nil): Essential food items (fresh vegetables, milk, eggs, bread, salt, unbranded rice/wheat), healthcare services, educational services. 5%: Essential items with some processing (sugar, edible oils, tea, coffee, coal, life-saving drugs, economy class air travel), non-AC restaurant services. 12%: Processed foods, computers, mobile phones, business class air travel, work contracts. 18%: Most goods and services — electronics, FMCG, restaurant services (AC), financial services, telecom, IT services. 28%: Luxury and sin goods — cars (above 1200cc petrol/1500cc diesel), cigarettes, aerated drinks, casinos, cement, large appliances. Special rates: 3% on gold/silver/precious stones, 0.25% on rough diamonds. GST on services follows similar slabs. As of 2024, the government is considering rationalizing slabs to reduce complexity."
    },
    {
        "instruction": "What is Input Tax Credit (ITC) in GST and what are the conditions to claim it?",
        "input": "A manufacturer purchases raw materials worth ₹5,00,000 + 18% GST. The finished goods are sold at ₹8,00,000 + 18% GST.",
        "output": "Input Tax Credit (ITC) allows a GST-registered business to deduct the GST paid on purchases (inputs) from the GST collected on sales (output). This prevents cascading taxation — like getting a credit for toll paid on one highway when paying on the next. Calculation: GST paid on inputs (ITC) = 18% × ₹5,00,000 = ₹90,000. GST collected on output = 18% × ₹8,00,000 = ₹1,44,000. Net GST payable = ₹1,44,000 - ₹90,000 = ₹54,000. Conditions to claim ITC: (1) Must be registered under GST, (2) Must have a valid tax invoice, (3) Goods/services received, (4) Supplier must have filed GSTR-1 and the invoice must appear in GSTR-2B of buyer, (5) Payment to supplier must be made within 180 days, (6) ITC claim in GSTR-3B within prescribed time. Blocked credits (Section 17(5)): ITC NOT available on: motor vehicles (for non-transport businesses), food/beverages, personal consumption, works contract for immovable property construction."
    },
    {
        "instruction": "What is the GST registration threshold and when is registration mandatory?",
        "input": "",
        "output": "GST registration is mandatory when aggregate turnover exceeds threshold limits: For goods suppliers: ₹40 lakhs per year (₹20 lakhs for special category states — Manipur, Mizoram, Nagaland, Tripura). For service providers: ₹20 lakhs per year (₹10 lakhs for special category states). For inter-state supply: Mandatory regardless of turnover (even ₹1 of inter-state supply requires GST registration). Mandatory registration regardless of turnover for: E-commerce operators, casual taxable persons, non-resident taxable persons, persons under reverse charge mechanism, input service distributors, agents supplying on behalf of others. Voluntary registration: Available even below threshold (beneficial for B2B businesses who want to pass ITC to clients). Timeline: Registration application must be filed within 30 days of crossing the threshold. The GSTIN (15-digit number) structure: 2 digits (state code) + 10 digits (PAN) + entity number + Z + check digit."
    },
    {
        "instruction": "What are the GST return filing deadlines for a regular taxpayer?",
        "input": "",
        "output": "For regular taxpayers (non-composition), the key GST returns and deadlines are: GSTR-1 (Outward supplies): Monthly filers (turnover >₹5 crore): 11th of next month. Quarterly filers (turnover <₹5 crore) using QRMP scheme: 13th of the month after the quarter end. GSTR-3B (Summary return + tax payment): Monthly filers: 20th of next month. Quarterly filers (QRMP): Last date of the month after the quarter. GSTR-9 (Annual return): 31st December of the next financial year. GSTR-9C (Reconciliation/Audit statement): Along with GSTR-9, mandatory if turnover >₹5 crore. Late filing penalties: ₹100 per day per act (CGST + SGST = ₹200/day), maximum ₹5,000. For nil returns: ₹20/day, maximum ₹500. Interest on late tax payment: 18% per annum under Section 50. Tip: File GSTR-1 before GSTR-3B — ITC claims of your buyers depend on your GSTR-1 being filed on time."
    },
    {
        "instruction": "What is the Reverse Charge Mechanism (RCM) under GST?",
        "input": "A GST-registered company pays ₹50,000 to a freelance advocate for legal services.",
        "output": "Under the Reverse Charge Mechanism (RCM), the GST liability shifts from the supplier to the recipient. Normally, the supplier collects and pays GST. Under RCM, the buyer pays GST directly to the government. Legal services from an advocate: This is a mandatory RCM service (Notification 13/2017). The company (recipient) must pay 18% GST on ₹50,000 = ₹9,000 directly to the government. The advocate does NOT charge or collect GST. Process: Company pays ₹50,000 to advocate (no GST in invoice), Company self-invoices (creates its own invoice for RCM liability), Company pays ₹9,000 GST under RCM in GSTR-3B, Company can then claim this ₹9,000 as ITC (if services are for business use). Other common RCM categories: Services from GTA (Goods Transport Agency), import of services, services from directors to a company, e-commerce purchases from unregistered suppliers. Cash flow impact: Unlike regular GST (ITC nets off), RCM requires cash outflow first, then ITC claim."
    },

    # ── SECTION 5: ITR FILING ────────────────────────────────────────
    {
        "instruction": "Which ITR form should a salaried employee with rental income file?",
        "input": "A government employee with salary income of ₹8 lakhs and rental income of ₹2.4 lakhs from one property.",
        "output": "A salaried employee with rental income should file ITR-1 (Sahaj) IF conditions are met, OR ITR-2 if not. ITR-1 (Sahaj) — eligible if: Total income up to ₹50 lakhs, Income from: salary/pension + one house property + other sources only, No capital gains income, Not a director in a company, No foreign assets. Since this taxpayer has salary + one house property rental income (both within ₹50L) → ITR-1 is applicable. Rental income calculation for ITR: Gross Annual Value (GAV) = ₹2,40,000. Less: Municipal taxes paid (if any). Net Annual Value (NAV) = remaining. Less: 30% standard deduction on NAV. Less: Home loan interest (if applicable, under Section 24(b)). Net income from house property = result (can be negative = loss). This is added to salary for tax computation. ITR-2 is needed if: Capital gains exist, More than one property, Foreign assets, Director of company, Income >₹50L."
    },
    {
        "instruction": "What is the due date for filing ITR and what is the penalty for late filing?",
        "input": "",
        "output": "Due dates for filing Income Tax Return (ITR) for FY 2024-25 (AY 2025-26): July 31, 2025: For individuals, HUF, firms (not requiring audit) — salaried, rental income, small business. October 31, 2025: For taxpayers requiring audit (business with turnover >₹1 crore for cash, >₹10 crore for non-cash). November 30, 2025: For taxpayers with international/specified domestic transactions (Transfer Pricing). December 31, 2025: Belated/revised returns. Penalty for late filing under Section 234F: ₹1,000 if total income ≤ ₹5 lakhs. ₹5,000 if total income >₹5 lakhs. No penalty if total income is below the basic exemption limit (₹3 lakhs under new regime, ₹2.5 lakhs under old). Additional consequences: Interest under Section 234A (1% per month on unpaid tax from due date). Loss carry-forward restriction: Business/capital losses (except house property loss) cannot be carried forward if ITR filed after due date. Revised return: Can be filed till December 31 of the assessment year."
    },

    # ── SECTION 6: OLD vs NEW TAX REGIME ────────────────────────────
    {
        "instruction": "What are the income tax slabs under the Old Tax Regime vs New Tax Regime for individuals below 60 years?",
        "input": "FY 2024-25 (AY 2025-26).",
        "output": "OLD TAX REGIME (with deductions like 80C, HRA, etc.): Up to ₹2,50,000: Nil. ₹2,50,001 - ₹5,00,000: 5%. ₹5,00,001 - ₹10,00,000: 20%. Above ₹10,00,000: 30%. Rebate under Section 87A: Up to ₹12,500 if total income ≤ ₹5 lakhs. NEW TAX REGIME (default from FY 2023-24, lower rates but no major deductions): Up to ₹3,00,000: Nil. ₹3,00,001 - ₹7,00,000: 5%. ₹7,00,001 - ₹10,00,000: 10%. ₹10,00,001 - ₹12,00,000: 15%. ₹12,00,001 - ₹15,00,000: 20%. Above ₹15,00,000: 30%. Rebate under Section 87A (New): Up to ₹25,000 if total income ≤ ₹7 lakhs (effectively zero tax up to ₹7L). Standard Deduction: ₹75,000 in new regime (increased from ₹50,000 in Budget 2024). Both regimes: Add 4% Health & Education Cess on tax amount. Surcharge: 10% if income 50L-1Cr, 15% if 1Cr-2Cr, 25% if 2Cr-5Cr, 25% if >5Cr (capped at 25% from 2023). How to choose: New regime wins if deductions < breakeven threshold (typically ₹3.75L for ₹15L income)."
    },
    {
        "instruction": "What is the advance tax payment schedule and when is it applicable?",
        "input": "",
        "output": "Advance tax is applicable when your estimated tax liability for the year exceeds ₹10,000 (after TDS). It prevents a large year-end tax burden — like paying EMIs instead of a lump sum. Payment schedule (4 installments): By June 15: 15% of estimated annual tax. By September 15: 45% of estimated annual tax (cumulative). By December 15: 75% of estimated annual tax (cumulative). By March 15: 100% of estimated annual tax (cumulative). Example: Estimated tax = ₹1,00,000. June 15: ₹15,000. September 15: ₹30,000 (₹45,000 total). December 15: ₹30,000 (₹75,000 total). March 15: ₹25,000 (₹1,00,000 total). Special rule for presumptive income (Section 44AD/44ADA): Pay 100% advance tax by March 15 (no quarterly installments required). Interest for non-payment: Section 234B: 1% per month if advance tax paid is <90% of assessed tax. Section 234C: 1% per month for shortfall in each installment. Senior citizens (60+) without business income: Exempt from advance tax."
    },

    # ── SECTION 7: GST COMPOSITION SCHEME ───────────────────────────
    {
        "instruction": "What is the GST Composition Scheme and who is eligible?",
        "input": "A small restaurant owner with annual turnover of ₹80 lakhs wants to simplify GST compliance.",
        "output": "The GST Composition Scheme is a simplified compliance mechanism for small businesses — like a flat-rate tax instead of regular GST calculations. Eligibility: Aggregate turnover ≤ ₹1.5 crore (₹75 lakhs for special category states). Not applicable for: inter-state suppliers, e-commerce operators, and manufacturers of notified goods. Tax rates under Composition Scheme: Manufacturers: 1% (0.5% CGST + 0.5% SGST) on turnover. Traders: 1% on turnover. Restaurants (not serving liquor): 5% on turnover (no ITC benefit). Service providers (Composition for services): 6% on turnover (notified scheme). Benefits: No detailed invoice-wise GST return, Pay fixed % of turnover instead of calculating output-ITC, File only 1 quarterly return (CMP-08) + 1 annual return (GSTR-4). Limitations: Cannot collect GST from customers (price must be inclusive), Cannot claim ITC on inputs, Cannot supply inter-state, Cannot supply exempt goods. For this restaurant owner (₹80L turnover): Pay 5% of ₹80L = ₹4L as GST annually. Much simpler than regular GST filing."
    },

    # ── SECTION 8: FORM 26AS & AIS ──────────────────────────────────
    {
        "instruction": "What is Form 26AS and how does it help in filing ITR?",
        "input": "",
        "output": "Form 26AS is your tax passbook — a consolidated annual tax statement maintained by the Income Tax Department showing all tax credits in your name. It contains: (1) TDS deducted by employers, banks, tenants, etc. (2) TCS (Tax Collected at Source) collected by sellers. (3) Advance tax paid by you. (4) Self-assessment tax paid. (5) Refunds issued. (6) High-value financial transactions (from 2021). How to access: Login at incometax.gov.in → My Account → View Form 26AS, OR via Net Banking of your bank (linked to PAN). How it helps in ITR filing: TDS entries in 26AS must match your ITR entries — any mismatch triggers a notice. You can claim credit ONLY for TDS shown in 26AS. If TDS deducted by employer/bank is not reflecting: Follow up with deductor to file/correct their TDS return. Annual Information Statement (AIS): A more comprehensive version (introduced 2021) showing financial transactions — mutual fund purchases, property deals, foreign remittances, etc. Always reconcile AIS with your records before filing ITR."
    },

    # ── SECTION 9: PRESUMPTIVE TAXATION ─────────────────────────────
    {
        "instruction": "What is presumptive taxation under Section 44AD for small businesses?",
        "input": "A small trader has annual turnover of ₹60 lakhs — mostly cash transactions.",
        "output": "Section 44AD provides presumptive taxation for eligible small businesses — a simplified scheme where income is presumed as a fixed percentage of turnover, without requiring detailed books of accounts. Eligibility: Resident individual, HUF, or partnership firm (not LLP), Engaged in any business (not commission agents, not eligible professions), Turnover ≤ ₹2 crore. Presumptive income rates: 8% of gross turnover (for cash receipts), 6% of gross turnover (for digital receipts — bank transfers, UPI, cheques). Calculation for this trader: Assuming 80% cash (₹48L) and 20% digital (₹12L). Presumptive income = (8% × ₹48L) + (6% × ₹12L) = ₹3.84L + ₹0.72L = ₹4.56L. This ₹4.56L is taxed at applicable income tax slab rates. Benefits: No books of accounts required (unless audited), No audit required (turnover <₹2 crore and income declared ≥ 6%/8%), Pay full advance tax by March 15 (single installment). Restriction: If you opt out in one year, you cannot use 44AD for the next 5 years without audit. File ITR-4 (Sugam) if using 44AD."
    },

    # ── SECTION 10: INTERNATIONAL TAX / NRI ─────────────────────────
    {
        "instruction": "What is the residential status test under the Income Tax Act and why does it matter?",
        "input": "An Indian software engineer went to work in the US on March 1, 2024 and did not return during FY 2024-25.",
        "output": "Residential status determines WHICH income is taxable in India — it's the foundation of international tax. Three residential statuses: Resident (and Ordinarily Resident - ROR): Taxable on global income. Resident but Not Ordinarily Resident (RNOR): Taxable on India-sourced income + income derived from a business/profession in India. Non-Resident (NR): Taxable ONLY on India-sourced income. Test for Resident status (Section 6): Condition 1: Present in India for ≥182 days in the financial year, OR Condition 2: Present in India for ≥60 days in the FY + ≥365 days in the preceding 4 FYs. For this engineer: Left India March 1, 2024 → Present for April 1 to March 1 = ~335 days... Wait: FY 2024-25 = April 1, 2024 to March 31, 2025. Engineer LEFT in March 2024, so was ALREADY abroad from April 1, 2024 onward. Days in India during FY 2024-25: 0 days → Does NOT satisfy 182-day condition → Non-Resident (NR). As NR: Only Indian salary (if any), Indian interest income, Indian property rental income, and Indian capital gains are taxable. US salary is NOT taxable in India."
    },
]

print(f"✅ Seed dataset: {len(TAX_QA_DATA)} Q&A pairs")
print(f"\nSample entry:\n")
print(f"INSTRUCTION: {TAX_QA_DATA[0]['instruction'][:80]}...")
print(f"INPUT: {TAX_QA_DATA[0]['input'][:60]}...")
print(f"OUTPUT: {TAX_QA_DATA[0]['output'][:120]}...")

✅ Seed dataset: 25 Q&A pairs

Sample entry:

INSTRUCTION: What is the TDS rate on professional fees under Section 194J and when is it appl...
INPUT: A company pays ₹45,000 to a freelance chartered accountant f...
OUTPUT: Under Section 194J of the Income Tax Act, TDS is deducted at 10% on professional fees paid to a resident. The threshold ...


In [ ]:
import openai
import json
import re # Added import

# ─── [OPTIONAL] Expand Dataset to 500 pairs using Claude/GPT-4 ──────
# This cell shows how to auto-generate more examples using an LLM.
# Skip this cell if you're using only the seed data (60 pairs) for a quick test run.

GENERATION_PROMPT = """
You are creating a training dataset for an Indian tax law AI assistant.
Respond ONLY with a JSON object. This object must have a single key, 'q_a_pairs', whose value is a JSON array containing {n} unique Q&A pairs about Indian tax law. Each entry in the 'q_a_pairs' array must have: instruction (question), input (optional context), output (detailed expert answer).

Rules for outputs:
- Always cite the relevant Income Tax Act section or GST notification
- Include specific thresholds, rates, and limits with ₹ amounts
- Show calculation steps if numbers are involved
- Mention compliance steps (filing, deadlines, forms)
- Write from a CA/tax practitioner perspective
- Never give generic answers — be specific to Indian tax law
"""

TOPICS_TO_GENERATE = [
    "TDS on salary under Section 192",
    "Home loan principal repayment under 80C",
    "GST on educational services",
    "LTCG on debt mutual funds post Budget 2023",
    "Section 44ADA for professionals",
    "GST e-invoicing requirements",
    "Tax on cryptocurrency gains in India",
    "Gift tax provisions under Income Tax Act",
    "Transfer pricing for related party transactions",
    "Faceless assessment scheme",
]

# Uncomment and use your preferred API:

client = openai.OpenAI(api_key="YOUR_API_KEYS")
for topic in TOPICS_TO_GENERATE:
    response = client.chat.completions.create(
        model="gpt-4o",
        max_tokens=4096,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": GENERATION_PROMPT.format(n=20, topics=topic)}]
    )
    raw_content = response.choices[0].message.content
    new_examples_dict = {}
    potential_qa_list = []

    try:
        new_examples_dict = json.loads(raw_content)
    except json.JSONDecodeError as e:
        print(f"Warning: JSONDecodeError on direct parse for topic '{topic}'. Error: {e}")
        print(f"Attempting regex extraction from raw content...")
        # Look for a JSON object or array, preferring the outermost one
        json_match = re.search(r'(\{[\s\S]*\})|(\[[\s\S]*\])', raw_content, re.DOTALL)
        if json_match:
            json_string = json_match.group(0) # Get the full matched string
            try:
                new_examples_dict = json.loads(json_string)
                print(f"Successfully extracted and parsed JSON via regex for topic '{topic}'.")
            except json.JSONDecodeError as e_inner:
                print(f"Error: JSONDecodeError on regex-extracted JSON for topic '{topic}'. Error: {e_inner}")
                print(f"Problematic content excerpt: {json_string[:500]}...")
                continue # Skip to the next topic if JSON is still invalid
        else:
            print(f"Warning: No valid JSON object or array found in LLM response for topic '{topic}'. Skipping.")
            print(f"Raw content: {raw_content[:500]}...")
            continue # Skip to the next topic if no JSON found

    # Now, parse new_examples_dict based on the expected structure from the new prompt
    if isinstance(new_examples_dict, dict) and 'q_a_pairs' in new_examples_dict:
        extracted_list = new_examples_dict['q_a_pairs']
        if isinstance(extracted_list, list) and all(isinstance(item, dict) for item in extracted_list):
            potential_qa_list = extracted_list
        else:
            print(f"Warning: 'q_a_pairs' value is not a list of dictionaries for topic '{topic}'. Skipping.")
            print(f"Value received: {extracted_list}")
    else:
        # Fallback for if LLM still returns a top-level array directly or another dict structure
        if isinstance(new_examples_dict, list):
            potential_qa_list = new_examples_dict
        elif isinstance(new_examples_dict, dict):
            # Check if it's a single Q&A dict at the top level
            if 'instruction' in new_examples_dict and 'output' in new_examples_dict:
                potential_qa_list = [new_examples_dict]
            else:
                # Look for a list of dicts in its values (e.g., {'Q&A': [...]})
                for value in new_examples_dict.values():
                    if isinstance(value, list) and all(isinstance(item, dict) for item in value):
                        potential_qa_list = value
                        break

    if potential_qa_list:
        # Ensure only dictionaries are extended, filtering out any potential non-dict items just in case
        TAX_QA_DATA.extend([item for item in potential_qa_list if isinstance(item, dict)])
    else:
        print(f"Warning: LLM returned an unrecognized structure or empty Q&A for topic '{topic}'. Skipping: {raw_content[:500]}...")

print("Dataset expansion prompt ready.")
print(f"Current dataset size: {len(TAX_QA_DATA)} pairs")
print("Uncomment the API section above and run to expand to 500+ pairs.")


Dataset expansion prompt ready.
Current dataset size: 226 pairs
Uncomment the API section above and run to expand to 500+ pairs.


In [ ]:
# ─── Format Dataset in Alpaca Prompt Template ──────────────────────
# Llama 3.1 uses the ChatML / Alpaca style prompting

ALPACA_PROMPT = """Below is an instruction that describes a tax law question. Write a response that accurately and completely answers the question.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token  # Must add EOS token at end of each training example

def format_alpaca(example):
    text = ALPACA_PROMPT.format(
        instruction=example["instruction"],
        input=example["input"] if example["input"] else "(No additional context)",
        output=example["output"]
    ) + EOS_TOKEN
    return {"text": text}

# Create HuggingFace Dataset
# Filter TAX_QA_DATA to ensure all elements are dictionaries with required keys and correct value types
cleaned_tax_qa_data = [
    item for item in TAX_QA_DATA
    if isinstance(item, dict)  # Ensure it's a dictionary
    and "instruction" in item and isinstance(item["instruction"], str)  # Ensure instruction key exists and its value is a string
    and "output" in item and isinstance(item["output"], str)        # Ensure output key exists and its value is a string
    and "input" in item and (isinstance(item["input"], str) or item["input"] is None) # Ensure input key exists and its value is a string or None
]
raw_dataset = Dataset.from_list(cleaned_tax_qa_data)
formatted_dataset = raw_dataset.map(format_alpaca, remove_columns=raw_dataset.column_names)

# Train/test split (90/10)
split = formatted_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"✅ Dataset formatted!")
print(f"Training examples: {len(train_dataset)}")
print(f"Evaluation examples: {len(eval_dataset)}")
print(f"\nSample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])


Map:   0%|          | 0/226 [00:00<?, ? examples/s]

✅ Dataset formatted!
Training examples: 203
Evaluation examples: 23

Sample formatted text (first 500 chars):
Below is an instruction that describes a tax law question. Write a response that accurately and completely answers the question.

### Instruction:
What are the provisions for tax deduction under Section 80D?

### Input:
Individual purchasing health insurance for family

### Response:
Under Section 80D of the Income Tax Act, deductions are allowed for premiums paid on health insurance policies. For individual, spouse, and dependent children, deduction up to ₹25,000 is available (₹50,000 for senio


In [ ]:
# ─── [OPTIONAL] Push Dataset to HuggingFace Hub ───────────────────
# This is optional but recommended — makes your dataset reusable and shareable

from huggingface_hub import login
login(token="YOUR_TOKENS")  # Get from huggingface.co/settings/tokens

#Push the raw (unformatted) dataset for reuse
raw_dataset.push_to_hub("keerthan222/indian-tax-law-qa", private=False)
print("✅ Dataset pushed to HuggingFace Hub!")

print("Dataset push ready (uncomment and add your HF token to push)")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 74.3kB / 74.3kB            

README.md:   0%|          | 0.00/345 [00:00<?, ?B/s]

✅ Dataset pushed to HuggingFace Hub!
Dataset push ready (uncomment and add your HF token to push)


---
## 🔥 Step 4: Fine-Tune with SFTTrainer

**The training loop analogy:** Imagine a student studying flashcards (the dataset) over multiple rounds (epochs).
Each pass, they get slightly better at answering tax questions. The loss value is like their error rate —
it should drop from ~2.0 (wild guessing) to ~0.6 (confident, accurate answers).

**Expected training time on Colab T4:** ~45 minutes for 54 examples × 3 epochs

In [ ]:
# ─── SFTTrainer Configuration ──────────────────────────────────────

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,  # Packing = True for short sequences; False for long answers like ours

    args = TrainingArguments(
        # ── Batch & Gradient ────────────────────────────
        per_device_train_batch_size = 2,         # 2 examples per GPU step
        gradient_accumulation_steps = 4,         # Effective batch size = 2 × 4 = 8

        # ── Training duration ───────────────────────────
        num_train_epochs = 6,                    # 6 full passes through dataset
        # max_steps = 100,                       # Alternative: fixed number of steps

        # ── Optimizer & Learning Rate ───────────────────
        learning_rate = 2e-4,                    # Standard for LoRA fine-tuning
        lr_scheduler_type = "cosine",            # Cosine decay (smooth learning rate reduction)
        warmup_ratio = 0.05,                     # 5% of steps for warmup
        optim = "adamw_8bit",                    # 8-bit AdamW saves VRAM

        # ── Memory optimization ─────────────────────────
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        gradient_checkpointing = True,           # Reduces VRAM at cost of ~20% speed

        # ── Logging & Saving ────────────────────────────
        logging_steps = 5,
        eval_steps = 20,
        save_steps = 50,
        output_dir = "./outputs/indian-tax-law",
        report_to = "none",                      # Change to "wandb" for experiment tracking

        # ── Misc ────────────────────────────────────────
        seed = 42,
        weight_decay = 0.01,
        max_grad_norm = 0.3,                     # Gradient clipping for stability
    ),
)

print("✅ SFTTrainer configured!")
print(f"Total training steps: {trainer.args.max_steps if trainer.args.max_steps > 0 else 'calculated from epochs'}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/203 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/23 [00:00<?, ? examples/s]

✅ SFTTrainer configured!
Total training steps: calculated from epochs


In [ ]:
# ─── Start Training ────────────────────────────────────────────────
# Monitor: training loss should decrease from ~2.0 → ~0.6
# If loss stalls above 1.5: dataset quality issue (review your examples)
# If loss drops too fast below 0.3: possible overfitting (add more diverse data)

import time
start_time = time.time()

# Check GPU memory before training
print(f"VRAM before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

trainer_stats = trainer.train()

elapsed = time.time() - start_time
print(f"\n🎉 Training complete!")
print(f"Time: {elapsed/60:.1f} minutes")
print(f"Final training loss: {trainer_stats.training_loss:.4f}")
print(f"Samples/second: {trainer_stats.metrics.get('train_samples_per_second', 'N/A')}")

VRAM before training: 5.92 GB / 15.6 GB


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 203 | Num Epochs = 6 | Total steps = 156
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,2.007490
10,1.404101
15,1.311135
20,1.066051
25,1.077563
30,0.873872
35,0.884858
40,0.860394
45,0.892711
50,0.865912


Unsloth: Restored added_tokens_decoder metadata in ./outputs/indian-tax-law/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/indian-tax-law/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/indian-tax-law/checkpoint-150/tokenizer_config.json.


Step,Training Loss
5,2.007490
10,1.404101
15,1.311135
20,1.066051
25,1.077563
30,0.873872
35,0.884858
40,0.860394
45,0.892711
50,0.865912


Unsloth: Restored added_tokens_decoder metadata in ./outputs/indian-tax-law/checkpoint-156/tokenizer_config.json.



🎉 Training complete!
Time: 17.9 minutes
Final training loss: 0.5596
Samples/second: 1.139


---
## 📊 Step 5: Evaluate — Base vs Fine-Tuned Model

We test both models on questions from our held-out evaluation set to measure improvement.
Think of this like giving the same exam to a student before and after a specialized course.

In [ ]:
# ─── Inference Helper ──────────────────────────────────────────────

def generate_answer(model, tokenizer, instruction, input_text="", max_new_tokens=512):
    """Generate an answer using the Alpaca prompt format."""
    FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

    prompt = ALPACA_PROMPT.format(
        instruction=instruction,
        input=input_text if input_text else "(No additional context)",
        output=""  # Leave blank — model will fill this in
    )

    inputs = tokenizer(
        [prompt],
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,       # Low temperature = more deterministic (good for factual answers)
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the response part (after "### Response:")
    response = generated.split("### Response:")[-1].strip()
    return response


print("✅ Inference helper ready")

✅ Inference helper ready


In [ ]:
# ─── Test Questions (held out from training) ────────────────────────

TEST_QUESTIONS = [
    {
        "instruction": "What is the Section 80G deduction for donations, and what are the qualifying charities?",
        "input": "A taxpayer donated ₹50,000 to a PM Relief Fund and ₹20,000 to a local NGO.",
        "expected_keywords": ["80G", "100%", "50%", "PM", "qualifying"],
    },
    {
        "instruction": "What is the GST rate on life insurance premium?",
        "input": "",
        "expected_keywords": ["18%", "GST", "insurance", "premium"],
    },
    {
        "instruction": "What is Section 44ADA presumptive taxation for professionals?",
        "input": "A freelance software developer has annual receipts of ₹40 lakhs.",
        "expected_keywords": ["44ADA", "50%", "professional", "₹50 lakh"],
    },
    {
        "instruction": "What is the tax treatment of cryptocurrency gains in India?",
        "input": "",
        "expected_keywords": ["30%", "Section 115BBH", "crypto", "VDA", "1%"],
    },
    {
        "instruction": "What is the penalty for not filing GST returns?",
        "input": "",
        "expected_keywords": ["₹200", "₹10,000", "GSTR", "penalty", "Section 47"],
    },
]

print(f"Evaluation set: {len(TEST_QUESTIONS)} held-out questions")
print("\nRunning inference on fine-tuned model...\n")

results = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"── Question {i}/{len(TEST_QUESTIONS)} ─────────────────────────")
    print(f"Q: {q['instruction'][:80]}")

    answer = generate_answer(model, tokenizer, q["instruction"], q["input"])

    # Simple keyword-based scoring
    keywords_found = [kw for kw in q["expected_keywords"] if kw.lower() in answer.lower()]
    score = len(keywords_found) / len(q["expected_keywords"])

    print(f"Answer (first 200 chars): {answer[:200]}...")
    print(f"Keywords found: {keywords_found} ({score*100:.0f}% accuracy)\n")

    results.append({"question": q["instruction"], "answer": answer, "score": score})

avg_score = sum(r["score"] for r in results) / len(results)
print(f"\n🎯 Average accuracy on test set: {avg_score*100:.1f}%")
print("(Target: >70% — base model typically achieves 30-40%)")

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluation set: 5 held-out questions

Running inference on fine-tuned model...

── Question 1/5 ─────────────────────────
Q: What is the Section 80G deduction for donations, and what are the qualifying cha


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

Answer (first 200 chars): Under Section 80G, donations to specified charities qualify for deductions up to 100% of the donation amount. The PM Relief Fund qualifies for 100% deduction under Section 80G(2)(x). The local NGO mus...
Keywords found: ['80G', '100%', 'PM'] (60% accuracy)

── Question 2/5 ─────────────────────────
Q: What is the GST rate on life insurance premium?


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer (first 200 chars): As per Notification No. 12/2017-Central Tax (Rate), life insurance premiums are exempt from GST under entry no. 84. This is a mandatory exemption and applies to all life insurance services....
Keywords found: ['GST', 'insurance', 'premium'] (75% accuracy)

── Question 3/5 ─────────────────────────
Q: What is Section 44ADA presumptive taxation for professionals?


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer (first 200 chars): Under Section 44ADA, professionals with gross receipts up to ₹50 lakhs can opt for presumptive taxation, declaring 50% as taxable income to simplify compliance. For this developer, using 44ADA would r...
Keywords found: ['44ADA', '50%', 'professional', '₹50 lakh'] (100% accuracy)

── Question 4/5 ─────────────────────────
Q: What is the tax treatment of cryptocurrency gains in India?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer (first 200 chars): As per the Income Tax Act, gains from the sale of cryptocurrency are taxed as capital gains, with the sale being treated as a capital asset under 'Other than Equity Shares'. Short-term capital gains (...
Keywords found: ['crypto'] (20% accuracy)

── Question 5/5 ─────────────────────────
Q: What is the penalty for not filing GST returns?
Answer (first 200 chars): As per Section 47 of the CGST Act, 2017, a late fee of ₹50 per day is charged for each day of delay in filing GSTR-3B or GSTR-1, up to a maximum of ₹5,000. For nil returns, the late fee is reduced to ...
Keywords found: ['GSTR', 'Section 47'] (40% accuracy)


🎯 Average accuracy on test set: 59.0%
(Target: >70% — base model typically achieves 30-40%)


---
## 💾 Step 6: Save Model & Export to GGUF

**GGUF (GPT-Generated Unified Format)** is the format used by Ollama and LM Studio for local inference.
Think of it like converting a movie to MP4 format — the same content, but universally compatible.

**Q4_K_M** = 4-bit quantization, medium quality. Best balance of size and accuracy for deployment.

In [ ]:
# ─── Save LoRA Adapters (lightweight — ~100MB) ─────────────────────
# This saves ONLY the trained adapter weights, not the full model.
# The adapter is later merged with the base model for deployment.

ADAPTER_SAVE_PATH = "./indian-tax-expert-lora"

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"✅ LoRA adapter saved to: {ADAPTER_SAVE_PATH}")
!du -sh {ADAPTER_SAVE_PATH}  # Show size

In [ ]:
# ─── Export to GGUF Format ─────────────────────────────────────────
# This merges the LoRA adapter into the base model and converts to GGUF.
# The q4_k_m quantization makes it ~4.5GB (vs 16GB for the original).

# GGUF_OUTPUT_PATH = "./indian-tax-expert-gguf"

# # Quantization options (trade-off: quality vs file size):
# # q2_k: Smallest (2.8GB), lowest quality
# # q4_k_m: Balanced (4.5GB) ← RECOMMENDED
# # q8_0: Highest quality (8.5GB), requires more RAM

# model.save_pretrained_gguf(
#     GGUF_OUTPUT_PATH,
#     tokenizer,
#     quantization_method="q4_k_m"
# )

# print(f"\n✅ GGUF model exported!")
# !ls -lh {GGUF_OUTPUT_PATH}/*.gguf

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/956 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in ./indian-tax-expert-gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [09:38<00:00, 144.72s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [05:45<00:00, 86.37s/it]


Unsloth: Merge process complete. Saved to `/content/indian-tax-expert-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['./indian-tax-expert-gguf_gguf/Meta-Llama-3.1-8B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 mi

In [ ]:
# ─── [OPTIONAL] Push to HuggingFace Hub ──────────────────────────
# This makes your model publicly available at:
# https://huggingface.co/keerthan222/indian-tax-expert-llama-3.1-8b

from huggingface_hub import login
login(token="YOUR_TOKENS")

#Push the LoRA adapter
model.push_to_hub("keerthan222/indian-tax-expert-llama-3.1-8b-lora")
tokenizer.push_to_hub("keerthan222/indian-tax-expert-llama-3.1-8b-lora")

#Push the GGUF model
# model.push_to_hub_gguf(
#     "keerthan222/indian-tax-expert-llama-3.1-8b-GGUF",
#     tokenizer,
#     quantization_method="q4_k_m"
# )

print("✅ Done! Adapter live at huggingface.co/keerthan222/indian-tax-expert-llama-3.1-8b-lora")

README.md:   0%|          | 0.00/579 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  556kB /  168MB            

Saved model to https://huggingface.co/keerthan222/indian-tax-expert-llama-3.1-8b-lora


...

Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_t_xlt6lz`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_t_xlt6lz_gguf/Meta-Llama-3.1-8B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_t_xlt6lz_gguf/Meta-Llama-3.1-8B-Instruct.Q4_K_M.gguf']

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1-8B-Instruct.Q4_K_M.gguf:   0%|          |  559kB / 4.92GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/keerthan222/indian-tax-expert-llama-3.1-8b-GGUF
Unsloth: Cleaning up temporary files...
HuggingFace Hub push ready


---
## 🚀 Step 7: Deploy Locally with Ollama

After downloading the GGUF file to your local machine, run these commands in your terminal:

```bash
# 1. Install Ollama (Mac/Linux)
curl -fsSL https://ollama.ai/install.sh | sh

# 2. Create a Modelfile
cat > Modelfile << 'EOF'
FROM ./indian-tax-expert-gguf/unsloth.Q4_K_M.gguf

SYSTEM """You are an expert Indian tax law assistant trained on the Income Tax Act 1961, GST laws, TDS provisions, and ITR filing procedures. You provide accurate, section-specific guidance with calculations and compliance steps. Always mention that for critical decisions, users should consult a qualified Chartered Accountant."""

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER num_predict 512
EOF

# 3. Create the Ollama model
ollama create indian-tax-expert -f Modelfile

# 4. Run it!
ollama run indian-tax-expert
```

You now have a **private, offline Indian tax assistant** running on your laptop!

---

## 🎯 What's Next: Stretch Goals

### 1. DPO Alignment (Day 3)
Create 200 preference pairs (chosen response vs rejected response for the same question):
```python
# Format for DPO:
{
    "prompt": "What is the GST rate on...",
    "chosen": "Accurate, section-cited answer...",
    "rejected": "Vague, incorrect answer without citations..."
}
```

### 2. RAG Layer (Day 4)
Add the actual Income Tax Act PDF as a vector knowledge base:
- Use LlamaIndex or LangChain
- Chunk the ITR Act by section
- At inference: retrieve relevant sections → inject into prompt → generate answer

### 3. Production API (Day 5)
Wrap the model in a FastAPI endpoint:
- Input: Tax question
- Output: Answer + cited section + confidence score
- Deploy on Hugging Face Spaces (free) or Railway

---
**Notebook complete! 🎉 You've built a domain-specific Indian Tax Law LLM from scratch.**

In [ ]:
# ─── Final Summary ─────────────────────────────────────────────────

print("="*60)
print("  INDIAN TAX LAW FINE-TUNING — COMPLETE SUMMARY")
print("="*60)
print(f"\n📊 Training Dataset: {len(TAX_QA_DATA)} Q&A pairs")
print(f"🤖 Base Model: Meta Llama 3.1 8B Instruct (4-bit)")
print(f"🔧 Method: QLoRA (rank=16, alpha=32)")
print(f"⚡ Trainable params: ~0.5% of total parameters")
print(f"📉 Final training loss: {trainer_stats.training_loss:.4f}")
print(f"\n📁 Outputs:")
print(f"  - LoRA adapter: {ADAPTER_SAVE_PATH}/")
print(f"  - GGUF model:   {GGUF_OUTPUT_PATH}/")
print(f"\n🚀 Deploy: ollama create indian-tax-expert -f Modelfile")
print(f"         ollama run indian-tax-expert")
print("\n⚠️  Reminder: This is a research tool. Verify all tax")
print("   advice against official CBDT notifications and")
print("   consult a qualified CA for critical decisions.")
print("="*60)

  INDIAN TAX LAW FINE-TUNING — COMPLETE SUMMARY

📊 Training Dataset: 226 Q&A pairs
🤖 Base Model: Meta Llama 3.1 8B Instruct (4-bit)
🔧 Method: QLoRA (rank=16, alpha=32)
⚡ Trainable params: ~0.5% of total parameters
📉 Final training loss: 0.5596

📁 Outputs:
  - LoRA adapter: ./indian-tax-expert-lora/
  - GGUF model:   ./indian-tax-expert-gguf/

🚀 Deploy: ollama create indian-tax-expert -f Modelfile
         ollama run indian-tax-expert

⚠️  Reminder: This is a research tool. Verify all tax
   advice against official CBDT notifications and
   consult a qualified CA for critical decisions.
